# 03 — BigQuery ML Analysis

After dbt has transformed the data into `opp_budget.fct_budget_execution`, run ML experiments:

1. **Linear regression**: predict next-year budget allocation
2. **K-means clustering**: group incisos by spending patterns

## Setup

In [ ]:
!pip install -q google-cloud-bigquery==3.27.0 db-dtypes==1.3.1 pyarrow==18.1.0 matplotlib==3.9.3

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = "fabled-imagery-488015-p6"
client = bigquery.Client(project=PROJECT_ID)

print(f"Connected to BigQuery project: {PROJECT_ID}")

## Preview Data

**Data completeness by source:**
- **2011-2019**: Complete (credits_resumen, ~1.8K rows/yr with execution)
- **2020, 2023**: Complete (CGN SIIF official totals, no category detail)
- **2021**: Partial (credits_2021, separate schema)
- **2005-2010, 2022, 2024**: Sparse (PDF extractions only)

ML models train on **complete years only** (2011-2020, 2023) to avoid bias from incomplete data.

In [ ]:
query = """
SELECT *
FROM `opp_budget.fct_budget_execution`
WHERE fiscal_year <= 2024
ORDER BY fiscal_year DESC, total_credito_vigente DESC
LIMIT 20
"""
df = client.query(query).to_dataframe()
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
df.head(20)

In [ ]:
# Define complete years (used across all ML cells)
COMPLETE_YEARS = "2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2023"

summary = client.query(f"""
SELECT
    'all_years' AS scope,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT fiscal_year) AS distinct_years,
    COUNT(DISTINCT inciso) AS distinct_incisos,
    MIN(fiscal_year) AS min_year,
    MAX(fiscal_year) AS max_year,
    SUM(total_credito_vigente) AS total_budget,
    SUM(total_ejecucion) AS total_ejecucion,
    SUM(total_inversion) AS total_inversion
FROM `opp_budget.fct_budget_execution`
WHERE fiscal_year <= 2024

UNION ALL

SELECT
    'complete_years' AS scope,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT fiscal_year) AS distinct_years,
    COUNT(DISTINCT inciso) AS distinct_incisos,
    MIN(fiscal_year) AS min_year,
    MAX(fiscal_year) AS max_year,
    SUM(total_credito_vigente) AS total_budget,
    SUM(total_ejecucion) AS total_ejecucion,
    SUM(total_inversion) AS total_inversion
FROM `opp_budget.fct_budget_execution`
WHERE fiscal_year IN ({COMPLETE_YEARS})
""").to_dataframe()

print("Data coverage comparison:")
summary

In [ ]:
import matplotlib.pyplot as plt

complete_set = {2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2023}

# Budget evolution by year — shade incomplete years
yearly = client.query("""
SELECT
    fiscal_year,
    SUM(total_credito_vigente) AS credito,
    SUM(total_ejecucion) AS ejecucion,
    SUM(total_inversion) AS inversion
FROM `opp_budget.fct_budget_execution`
WHERE fiscal_year <= 2024
GROUP BY fiscal_year
ORDER BY fiscal_year
""").to_dataframe()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly["fiscal_year"], yearly["credito"] / 1e9, marker="o", label="Credito Vigente")
ax.plot(yearly["fiscal_year"], yearly["ejecucion"] / 1e9, marker="s", label="Ejecucion")
ax.plot(yearly["fiscal_year"], yearly["inversion"] / 1e9, marker="^", label="Inversion")

# Shade incomplete years
for _, row in yearly.iterrows():
    yr = int(row["fiscal_year"])
    if yr not in complete_set:
        ax.axvspan(yr - 0.4, yr + 0.4, alpha=0.15, color="red")

ax.set_xlabel("Fiscal Year")
ax.set_ylabel("Billions (UYU)")
ax.set_title("Uruguay National Budget (2005-2024) — Red shading = incomplete data")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Top 10 incisos by total budget (complete years only)
top_incisos = client.query(f"""
SELECT
    inciso,
    denominacion_inciso,
    SUM(total_credito_vigente) AS total_budget
FROM `opp_budget.fct_budget_execution`
WHERE fiscal_year IN ({COMPLETE_YEARS})
GROUP BY inciso, denominacion_inciso
ORDER BY total_budget DESC
LIMIT 10
""").to_dataframe()

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(
    [f"{r.inciso} - {r.denominacion_inciso[:30]}" for _, r in top_incisos.iterrows()],
    top_incisos["total_budget"] / 1e9,
)
ax.set_xlabel("Total Budget (Billions UYU)")
ax.set_title("Top 10 Government Agencies by Total Budget (complete years only)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Experiment 1: Linear Regression — Budget Forecasting

In [ ]:
# Linear regression: train only on complete years, aggregated to inciso level
# Government budgets are autoregressive: last year's budget predicts this year's
create_model_sql = f"""
CREATE OR REPLACE MODEL `opp_budget.budget_forecast`
OPTIONS(
    model_type='LINEAR_REG',
    input_label_cols=['total_credito']
) AS
WITH inciso_yearly AS (
    SELECT
        fiscal_year,
        inciso,
        SUM(total_credito_vigente) AS total_credito,
        SUM(total_ejecucion) AS total_ejecucion,
        SUM(total_inversion) AS total_inversion
    FROM `opp_budget.fct_budget_execution`
    WHERE fiscal_year IN ({COMPLETE_YEARS})
    GROUP BY fiscal_year, inciso
),
lagged AS (
    SELECT
        fiscal_year,
        inciso,
        total_credito,
        LAG(total_credito) OVER (
            PARTITION BY inciso ORDER BY fiscal_year
        ) AS prev_year_credito,
        LAG(total_ejecucion) OVER (
            PARTITION BY inciso ORDER BY fiscal_year
        ) AS prev_year_ejecucion,
        LAG(total_inversion) OVER (
            PARTITION BY inciso ORDER BY fiscal_year
        ) AS prev_year_inversion
    FROM inciso_yearly
)
SELECT
    fiscal_year,
    inciso,
    prev_year_credito,
    COALESCE(prev_year_ejecucion, 0) AS prev_year_ejecucion,
    COALESCE(prev_year_inversion, 0) AS prev_year_inversion,
    total_credito
FROM lagged
WHERE prev_year_credito IS NOT NULL
"""

print(f"Training on complete years only: {COMPLETE_YEARS}")
print("Aggregated to inciso level, lag-based features...")
client.query(create_model_sql).result()
print("Model created: opp_budget.budget_forecast")

In [ ]:
# Evaluate
eval_df = client.query("""
SELECT * FROM ML.EVALUATE(MODEL `opp_budget.budget_forecast`)
""").to_dataframe()
print("Model evaluation metrics:")
eval_df

In [ ]:
# Predict using last complete year (2023) as base → forecast 2024
# We can then compare predictions vs actual 2020 data for validation
PREDICT_BASE_YEAR = 2020  # last complete year before a gap — predict 2023 and compare

# Backtest: use 2019 actuals to predict 2020, then compare with CGN official 2020
print(f"Backtesting: predict {PREDICT_BASE_YEAR} using {PREDICT_BASE_YEAR - 1} actuals\n")

predict_df = client.query(f"""
WITH inciso_base AS (
    SELECT
        inciso,
        SUM(total_credito_vigente) AS total_credito,
        SUM(total_ejecucion) AS total_ejecucion,
        SUM(total_inversion) AS total_inversion
    FROM `opp_budget.fct_budget_execution`
    WHERE fiscal_year = {PREDICT_BASE_YEAR - 1}
    GROUP BY inciso
),
inciso_actual AS (
    SELECT
        inciso,
        SUM(total_credito_vigente) AS actual_credito
    FROM `opp_budget.fct_budget_execution`
    WHERE fiscal_year = {PREDICT_BASE_YEAR}
    GROUP BY inciso
)
SELECT
    p.inciso,
    d.denominacion_inciso,
    ROUND(p.predicted_total_credito, 0) AS predicted_budget,
    ROUND(a.actual_credito, 0) AS actual_budget,
    ROUND(ABS(p.predicted_total_credito - a.actual_credito) / a.actual_credito * 100, 1) AS error_pct
FROM ML.PREDICT(MODEL `opp_budget.budget_forecast`,
    (SELECT
        {PREDICT_BASE_YEAR} AS fiscal_year,
        inciso,
        total_credito AS prev_year_credito,
        COALESCE(total_ejecucion, 0) AS prev_year_ejecucion,
        COALESCE(total_inversion, 0) AS prev_year_inversion,
        total_credito
    FROM inciso_base)
) p
LEFT JOIN `opp_budget.dim_incisos` d ON p.inciso = d.inciso
LEFT JOIN inciso_actual a ON p.inciso = a.inciso
WHERE a.actual_credito IS NOT NULL AND a.actual_credito > 0
ORDER BY a.actual_credito DESC
LIMIT 15
""").to_dataframe()
print(f"Backtest: {PREDICT_BASE_YEAR - 1} → {PREDICT_BASE_YEAR} (predicted vs actual, top 15):")
predict_df

In [ ]:
# Backtest visualization: predicted vs actual
plot_df = predict_df.head(10).copy()
fig, ax = plt.subplots(figsize=(12, 6))
labels = [f"{r.inciso} - {str(r.denominacion_inciso)[:30]}" for _, r in plot_df.iterrows()]
x = range(len(labels))
width = 0.35
ax.barh([i - width/2 for i in x], plot_df["actual_budget"] / 1e9, width, label=f"Actual {PREDICT_BASE_YEAR}")
ax.barh([i + width/2 for i in x], plot_df["predicted_budget"] / 1e9, width, label=f"Predicted {PREDICT_BASE_YEAR}")
ax.set_yticks(list(x))
ax.set_yticklabels(labels)
ax.set_xlabel("Budget (Billions UYU)")
ax.set_title(f"Backtest: {PREDICT_BASE_YEAR} Predicted vs Actual (Top 10 Incisos)")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Error summary
median_err = predict_df["error_pct"].median()
mean_err = predict_df["error_pct"].mean()
print(f"\nBacktest error: median {median_err:.1f}%, mean {mean_err:.1f}%")

## Experiment 2: K-Means Clustering — Spending Patterns

In [ ]:
# K-Means clustering — complete years only, aggregated per inciso
kmeans_sql = f"""
CREATE OR REPLACE MODEL `opp_budget.inciso_clusters`
OPTIONS(
    model_type='KMEANS',
    num_clusters=4
) AS
SELECT
    f.inciso,
    AVG(f.total_credito_vigente) AS avg_budget,
    AVG(f.total_ejecucion) AS avg_execution,
    AVG(LEAST(COALESCE(f.avg_execution_rate_pct, 0), 100)) AS avg_exec_rate,
    AVG(f.total_inversion) AS avg_inversion,
    COUNT(DISTINCT f.fiscal_year) AS years_active
FROM `opp_budget.fct_budget_execution` f
WHERE f.fiscal_year IN ({COMPLETE_YEARS})
  AND f.total_credito_vigente IS NOT NULL
  AND f.total_credito_vigente > 0
GROUP BY f.inciso
"""

print(f"Training K-Means on complete years: {COMPLETE_YEARS}")
client.query(kmeans_sql).result()
print("Model created: opp_budget.inciso_clusters")

In [ ]:
# Cluster assignments with canonical names from dim_incisos
clusters_df = client.query(f"""
SELECT
    p.CENTROID_ID,
    p.inciso,
    d.denominacion_inciso,
    p.avg_budget,
    p.avg_execution,
    p.avg_exec_rate,
    p.avg_inversion
FROM ML.PREDICT(MODEL `opp_budget.inciso_clusters`,
    (SELECT
        f.inciso,
        AVG(f.total_credito_vigente) AS avg_budget,
        AVG(f.total_ejecucion) AS avg_execution,
        AVG(LEAST(COALESCE(f.avg_execution_rate_pct, 0), 100)) AS avg_exec_rate,
        AVG(f.total_inversion) AS avg_inversion,
        COUNT(DISTINCT f.fiscal_year) AS years_active
    FROM `opp_budget.fct_budget_execution` f
    WHERE f.fiscal_year IN ({COMPLETE_YEARS})
      AND f.total_credito_vigente IS NOT NULL
      AND f.total_credito_vigente > 0
    GROUP BY f.inciso)
) p
LEFT JOIN `opp_budget.dim_incisos` d ON p.inciso = d.inciso
ORDER BY CENTROID_ID, avg_budget DESC
""").to_dataframe()

print(f"Clusters: {clusters_df['CENTROID_ID'].nunique()}, Incisos: {len(clusters_df)}")
clusters_df

In [ ]:
# Cluster summary with agency names
print("Cluster profiles:\n")
for cid in sorted(clusters_df["CENTROID_ID"].unique()):
    group = clusters_df[clusters_df["CENTROID_ID"] == cid]
    names = [f"{r.inciso}-{str(r.denominacion_inciso)[:35]}" for _, r in group.iterrows()]
    print(f"  Cluster {cid}: {len(group)} incisos")
    print(f"    Avg budget:     {group['avg_budget'].mean():>15,.0f} UYU")
    print(f"    Avg execution:  {group['avg_execution'].mean():>15,.0f} UYU")
    print(f"    Avg exec rate:  {group['avg_exec_rate'].mean():>8.1f}%")
    print(f"    Avg investment: {group['avg_inversion'].mean():>15,.0f} UYU")
    print(f"    Agencies:       {', '.join(names[:5])}")
    if len(names) > 5:
        print(f"                    ... and {len(names) - 5} more")
    print()

In [ ]:
# Scatter plot of clusters: avg budget vs exec rate
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for cid in sorted(clusters_df["CENTROID_ID"].unique()):
    group = clusters_df[clusters_df["CENTROID_ID"] == cid]
    ax.scatter(
        group["avg_budget"] / 1e9,
        group["avg_exec_rate"],
        c=colors[cid - 1] if cid <= len(colors) else "gray",
        label=f"Cluster {cid} ({len(group)} incisos)",
        s=100, alpha=0.7, edgecolors="black", linewidth=0.5,
    )
    # Label each point with inciso number
    for _, r in group.iterrows():
        ax.annotate(str(r.inciso), (r.avg_budget / 1e9, r.avg_exec_rate),
                     fontsize=7, ha="center", va="bottom")
ax.set_xlabel("Avg Budget (Billions UYU)")
ax.set_ylabel("Avg Execution Rate (%, capped at 100)")
ax.set_title("K-Means Clusters: Budget Size vs Execution Rate (2005-2024)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("BQML analysis complete.")
print("Models created:")
print("  - opp_budget.budget_forecast  (LINEAR_REG)")
print("  - opp_budget.inciso_clusters  (KMEANS)")